In [45]:
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import os

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using {device} device!")

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
Using cuda device!


In [46]:
print("Training size:", len(training_data))
print("Test size:", len(test_data))

x0, y0 = training_data[0]
print("One image shape:", x0.shape)
print("One label:", y0, classes[y0])

images = training_data.data.float() / 255.0
labels = torch.as_tensor(training_data.targets)

print("Pixel min:", images.min().item())
print("Pixel max:", images.max().item())
print("Pixel mean:", images.mean().item())
print("Pixel std:", images.std().item())

counts = torch.bincount(labels, minlength=10)

for i, count in enumerate(counts):
    print(i, classes[i], count.item())

Training size: 60000
Test size: 10000
One image shape: torch.Size([1, 28, 28])
One label: 9 Ankle boot


Pixel min: 0.0
Pixel max: 1.0
Pixel mean: 0.2860405743122101
Pixel std: 0.3530242443084717
0 T-shirt/top 6000
1 Trouser 6000
2 Pullover 6000
3 Dress 6000
4 Coat 6000
5 Sandal 6000
6 Shirt 6000
7 Sneaker 6000
8 Bag 6000
9 Ankle boot 6000


In [47]:
class BalancedFashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 28x28 -> 14x14

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 14x14 -> 7x7

            # Block 3, small spatial size so still fast
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)

In [48]:
torch.manual_seed(307)

batch_size = 256

train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

model = BalancedFashionCNN().to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 8

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader)
    test_acc = get_accuracy(model, test_loader)
    print(f"Epoch {epoch + 1}: loss = {train_loss:.4f}, test accuracy = {test_acc:.4f}")

Epoch 1: loss = 0.4014, test accuracy = 0.8845
Epoch 2: loss = 0.2604, test accuracy = 0.9028
Epoch 3: loss = 0.2169, test accuracy = 0.9131
Epoch 4: loss = 0.1892, test accuracy = 0.9170
Epoch 5: loss = 0.1709, test accuracy = 0.9209
Epoch 6: loss = 0.1538, test accuracy = 0.9200
Epoch 7: loss = 0.1403, test accuracy = 0.9216
Epoch 8: loss = 0.1239, test accuracy = 0.9207


In [54]:
import os
import torch

model = model.to("cpu")
model.eval()

# Safer TorchScript save
model_scripted = torch.jit.script(model)
model_scripted.save("fashion.pt")

file_size_mb = os.path.getsize("fashion.pt") / (1024 * 1024)
print("Saved file size:", file_size_mb, "MB")

Saved file size: 1.7816295623779297 MB


In [55]:
loaded_model = torch.jit.load("fashion.pt", map_location="cpu")
loaded_model.eval()

# Test on CPU, because PrairieLearn likely grades on CPU
device_test = "cpu"

def get_accuracy_cpu(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device_test)
            y = y.to(device_test)

            pred = model(X)
            predicted_class = pred.argmax(dim=1)

            correct += (predicted_class == y).sum().item()
            total += y.size(0)

    return correct / total

loaded_acc = get_accuracy_cpu(loaded_model, test_loader)
print("Loaded model test accuracy:", loaded_acc)

Loaded model test accuracy: 0.9207
